# Loan Counterfactual Analysis with DiCE

loan の与信モデルに対して反実仮想を生成する。`DiCE` を使い、生データ空間で判定反転に必要な変化を確認する。

## 前提

探索と表示はどちらも生データ空間で行う。

## 依存関係

```python
%pip install pandas numpy scikit-learn joblib dice-ml pyarrow
```

In [68]:
%pip install pandas numpy scikit-learn joblib dice-ml pyarrow plotly

Note: you may need to restart the kernel to use updated packages.


In [69]:
import dice_ml

`dice_ml` の import に失敗する場合は、上の `%pip install ...` を先に実行する。

In [70]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from IPython.display import display

In [71]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "model"

TRAIN_PATH = DATA_DIR / "train_df.parquet"
TEST_PATH = DATA_DIR / "test_df.parquet"
ARTIFACT_PATH = MODEL_DIR / "loan_xgb_artifact.joblib"

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)
artifact = joblib.load(ARTIFACT_PATH)

model = artifact["model"]
artifact_feature_cols = artifact["feature_cols"]
state_feature_cols = [col for col in artifact_feature_cols if col.startswith("state_")]
feature_cols = [col for col in artifact_feature_cols if col not in state_feature_cols]
target_col = artifact["target_col"]
class_names = artifact["class_names"]
class_mapping = artifact["class_mapping"]
best_threshold = artifact["best_threshold"]

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("feature count in artifact:", len(artifact_feature_cols))
print("feature count used in counterfactual analysis:", len(feature_cols))
print("excluded state feature count:", len(state_feature_cols))
print("target column:", target_col)
print("class mapping:", class_mapping)
print(f"best threshold: {best_threshold:.4f}")

train shape: (236725, 56)
test shape: (59182, 56)
feature count in artifact: 4
feature count used in counterfactual analysis: 4
excluded state feature count: 0
target column: is_rejected
class mapping: {np.int64(0): np.int64(0), np.int64(1): np.int64(1)}
best threshold: 0.4200


In [72]:
missing_in_train = sorted(set(feature_cols + [target_col]) - set(train_df.columns))
missing_in_test = sorted(set(feature_cols + [target_col]) - set(test_df.columns))

if missing_in_train or missing_in_test:
    raise ValueError(
        f"Missing columns. train={missing_in_train}, test={missing_in_test}"
    )

model_train_df = train_df[feature_cols + [target_col]].copy()
model_test_df = test_df[feature_cols + [target_col]].copy()

model_train_scaled_df = model_train_df[feature_cols].copy()
model_test_scaled_df = model_test_df[feature_cols].copy()

model_train_scaled_df[target_col] = model_train_df[target_col].values
model_test_scaled_df[target_col] = model_test_df[target_col].values

display(model_train_df.head())
display(model_train_scaled_df.head())

,FICO_risk_score,dti_log1p,employment_length_ordinal,requested_loan_amount,is_rejected
0,622.0,3.335058,0.0,25000.0,1
1,732.0,2.544747,8.0,11125.0,0
2,672.0,3.608212,0.0,15000.0,1
3,727.0,4.387387,7.0,14500.0,1
4,741.0,2.578701,9.0,14000.0,1


,FICO_risk_score,dti_log1p,employment_length_ordinal,requested_loan_amount,is_rejected
0,622.0,3.335058,0.0,25000.0,1
1,732.0,2.544747,8.0,11125.0,0
2,672.0,3.608212,0.0,15000.0,1
3,727.0,4.387387,7.0,14500.0,1
4,741.0,2.578701,9.0,14000.0,1


## 介入特徴

特殊値フラグは固定し、介入特徴は 3 列に絞る。

In [73]:
actionable_features = [
    "dti_log1p",
    "employment_length_ordinal",
    "requested_loan_amount",
]

print("actionable feature count:", len(actionable_features))
print(actionable_features)

actionable feature count: 3
['dti_log1p', 'employment_length_ordinal', 'requested_loan_amount']


## モデル

`DiCE` とモデルの入力空間を生データのまま揃える。

In [74]:
class DiceReadyModel:
    def __init__(self, model, feature_names, threshold):
        self.model = model
        self.feature_names = feature_names
        self.threshold = threshold
        self.classes_ = np.array([0, 1])

    def _prepare(self, input_df: pd.DataFrame) -> pd.DataFrame:
        return input_df[self.feature_names].copy()

    def predict_proba(self, input_df: pd.DataFrame) -> np.ndarray:
        X = self._prepare(input_df)
        return self.model.predict_proba(X)

    def predict(self, input_df: pd.DataFrame) -> np.ndarray:
        positive_proba = self.predict_proba(input_df)[:, 1]
        return (positive_proba >= self.threshold).astype(int)


dice_ready_model = DiceReadyModel(
    model=model,
    feature_names=feature_cols,
    threshold=best_threshold,
)


def inverse_scale_features(input_df: pd.DataFrame) -> pd.DataFrame:
    return input_df[feature_cols].copy()

In [75]:
scored_test_df = model_test_df.copy()
scored_test_df["risk_proba"] = dice_ready_model.predict_proba(scored_test_df[feature_cols])[:, 1]
scored_test_df["predicted_risk"] = dice_ready_model.predict(scored_test_df[feature_cols])

display(scored_test_df[[target_col, "risk_proba", "predicted_risk"]].head())
print("share predicted as risky in test:", scored_test_df["predicted_risk"].mean())

,is_rejected,risk_proba,predicted_risk
0,0,0.225732,0
1,1,0.999156,1
2,1,0.995119,1
3,0,0.067599,0
4,0,0.081740,0


share predicted as risky in test: 0.49731337230914807


## DiCE 設定

`method="genetic"` を使い、学習分布は生データで与える。

In [76]:
dice_data = dice_ml.Data(
    dataframe=model_train_df,
    continuous_features=feature_cols,
    outcome_name=target_col,
)

dice_model = dice_ml.Model(
    model=dice_ready_model,
    backend="sklearn",
    model_type="classifier",
)

dice = dice_ml.Dice(dice_data, dice_model, method="genetic")

In [77]:
# model_input_feature_meanings.txt に合わせて、
# 介入対象 3 列だけに意味ベースの制約を与える。
# dti_log1p と requested_loan_amount は小さいほど望ましい方向、
# employment_length_ordinal は大きいほど望ましい方向として扱う。

monotone_decrease_features = [
    "dti_log1p",
    "requested_loan_amount",
]

monotone_increase_features = [
    "employment_length_ordinal",
]

domain_bounds = {
    "dti_log1p": (0.0, None),
    "employment_length_ordinal": (0.0, 10.0),
    "requested_loan_amount": (0.0, None),
}


def build_permitted_range(df: pd.DataFrame, query_df: pd.DataFrame, columns: list[str]) -> dict:
    permitted = {}
    query_row = query_df.iloc[0]

    for col in columns:
        lower = float(df[col].min())
        upper = float(df[col].max())

        if col in domain_bounds:
            domain_lower, domain_upper = domain_bounds[col]
            if domain_lower is not None:
                lower = max(lower, float(domain_lower))
            if domain_upper is not None:
                upper = min(upper, float(domain_upper))

        if col in monotone_decrease_features:
            upper = float(query_row[col])

        if col in monotone_increase_features:
            lower = float(query_row[col])

        if upper < lower:
            upper = lower

        permitted[col] = [lower, upper]

    return permitted


def scale_feature_value(feature_name: str, value: float) -> float:
    return float(value)


In [78]:
def summarize_changes(query_df: pd.DataFrame, cf_df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    base_row = query_df.iloc[0]
    records = []

    for cf_idx, (_, cf_row) in enumerate(cf_df.iterrows(), start=1):
        for col in columns:
            before = float(base_row[col])
            after = float(cf_row[col])
            if np.isclose(before, after):
                continue
            records.append(
                {
                    "cf_id": f"CF_{cf_idx}",
                    "feature": col,
                    "before": before,
                    "after": after,
                    "delta": after - before,
                    "abs_delta": abs(after - before),
                }
            )

    return pd.DataFrame(records)


def build_scenario_value_matrices(
    query_unscaled: pd.DataFrame,
    cf_unscaled: pd.DataFrame,
    query_scaled: pd.DataFrame,
    cf_scaled: pd.DataFrame,
    columns: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    scenario_labels = ["original"] + [f"CF_{i}" for i in range(1, len(cf_unscaled) + 1)]

    raw_matrix = pd.concat(
        [query_unscaled[columns], cf_unscaled[columns]],
        axis=0,
    )
    raw_matrix.index = scenario_labels

    scaled_matrix = pd.concat(
        [query_scaled[columns], cf_scaled[columns]],
        axis=0,
    )
    scaled_matrix.index = scenario_labels

    return raw_matrix, scaled_matrix


def style_raw_value_matrix(raw_matrix: pd.DataFrame, scaled_matrix: pd.DataFrame, caption: str = ""):
    scale_max = float(np.abs(scaled_matrix.to_numpy()).max())
    if scale_max == 0:
        scale_max = 1.0

    styler = raw_matrix.style.format("{:.4f}").background_gradient(
        cmap="RdBu_r",
        axis=None,
        gmap=scaled_matrix,
        vmin=-scale_max,
        vmax=scale_max,
    )
    if caption:
        styler = styler.set_caption(caption)
    return styler


def plot_scenario_value_dotplot(
    raw_matrix: pd.DataFrame,
    scaled_matrix: pd.DataFrame,
    title: str,
    width: int = 1100,
    height_per_feature: int = 55,
):
    plot_df = scaled_matrix.loc[:, raw_matrix.columns].T.iloc[::-1]
    raw_plot_df = raw_matrix.loc[:, raw_matrix.columns].T.iloc[::-1]
    original_scaled = plot_df["original"]
    original_raw = raw_plot_df["original"]
    delta_df = plot_df.subtract(original_scaled, axis=0)
    raw_delta_df = raw_plot_df.subtract(original_raw, axis=0)
    scenario_names = list(delta_df.columns)
    feature_names = list(delta_df.index)
    axis_limit = float(np.abs(delta_df.to_numpy()).max())
    if axis_limit == 0:
        axis_limit = 1.0

    palette = [
        "#1f77b4",
        "#ff7f0e",
        "#2ca02c",
        "#d62728",
        "#9467bd",
        "#8c564b",
    ]
    symbols = ["circle", "square", "diamond", "triangle-up", "cross", "x"]

    fig = go.Figure()
    for idx, scenario in enumerate(scenario_names):
        fig.add_trace(
            go.Scatter(
                x=delta_df[scenario].values,
                y=feature_names,
                mode="markers",
                name=scenario,
                marker={
                    "size": 10,
                    "color": palette[idx % len(palette)],
                    "symbol": symbols[idx % len(symbols)],
                    "line": {"color": "#ffffff", "width": 1},
                },
                customdata=np.column_stack([
                    original_raw.values,
                    raw_plot_df[scenario].values,
                    raw_delta_df[scenario].values,
                    delta_df[scenario].values,
                    np.array([scenario] * len(feature_names), dtype=object),
                    np.array(feature_names, dtype=object),
                ]),
                hovertemplate=(
                    "scenario=%{customdata[4]}<br>"
                    "feature=%{customdata[5]}<br>"
                    "original raw=%{customdata[0]:.4f}<br>"
                    "scenario raw=%{customdata[1]:.4f}<br>"
                    "raw delta=%{customdata[2]:.4f}<br>"
                    "standardized delta=%{customdata[3]:.4f}<extra></extra>"
                ),
            )
        )

    fig.add_vline(x=0, line_dash="dash", line_color="#6e7781", line_width=1)
    fig.update_layout(
        title=title,
        width=width,
        height=max(350, height_per_feature * len(feature_names)),
        margin={"l": 260, "r": 40, "t": 70, "b": 60},
        template="plotly_white",
        hovermode="closest",
        xaxis={"title": "Standardized delta from original", "range": [-axis_limit, axis_limit]},
        yaxis_title="",
        legend_title_text="Scenario",
    )
    fig.update_yaxes(categoryorder="array", categoryarray=feature_names, automargin=True)
    fig.show()


def plot_scenario_value_barplot(
    raw_matrix: pd.DataFrame,
    scaled_matrix: pd.DataFrame,
    title: str,
    width: int = 1200,
    height_per_feature: int = 65,
):
    plot_df = raw_matrix.T.iloc[::-1]
    scaled_plot_df = scaled_matrix.loc[:, raw_matrix.columns].T.iloc[::-1]
    scenario_names = list(plot_df.columns)
    feature_names = list(plot_df.index)

    palette = [
        "#1f77b4",
        "#ff7f0e",
        "#2ca02c",
        "#d62728",
        "#9467bd",
        "#8c564b",
    ]

    fig = go.Figure()
    for idx, scenario in enumerate(scenario_names):
        fig.add_trace(
            go.Bar(
                x=plot_df[scenario].values,
                y=feature_names,
                name=scenario,
                orientation="h",
                marker_color=palette[idx % len(palette)],
                customdata=np.column_stack([
                    plot_df[scenario].values,
                    scaled_plot_df[scenario].values,
                    np.array([scenario] * len(feature_names), dtype=object),
                    np.array(feature_names, dtype=object),
                ]),
                hovertemplate=(
                    "scenario=%{customdata[2]}<br>"
                    "feature=%{customdata[3]}<br>"
                    "raw=%{customdata[0]:.4f}<br>"
                    "standardized=%{customdata[1]:.4f}<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        title=title,
        width=width,
        height=max(380, height_per_feature * len(feature_names)),
        template="plotly_white",
        barmode="group",
        hovermode="closest",
        xaxis_title="Raw value",
        yaxis_title="",
        legend_title_text="Scenario",
    )
    fig.update_yaxes(categoryorder="array", categoryarray=feature_names)
    fig.show()


def plot_scenario_radar(
    raw_matrix: pd.DataFrame,
    scaled_matrix: pd.DataFrame,
    title: str,
    width: int = 850,
    height: int = 850,
):
    plot_df = scaled_matrix.loc[:, raw_matrix.columns].T.iloc[::-1]
    raw_plot_df = raw_matrix.loc[:, raw_matrix.columns].T.iloc[::-1]
    scenario_names = list(plot_df.columns)
    feature_names = list(plot_df.index)
    theta = feature_names + [feature_names[0]]
    radial_limit = max(1.0, float(np.abs(plot_df.to_numpy()).max()))

    palette = ["#111111", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]

    fig = go.Figure()
    for idx, scenario in enumerate(scenario_names):
        scaled_values = plot_df[scenario].tolist()
        raw_values = raw_plot_df[scenario].tolist()
        fig.add_trace(
            go.Scatterpolar(
                r=scaled_values + [scaled_values[0]],
                theta=theta,
                mode="lines+markers",
                name=scenario,
                line={"color": palette[idx % len(palette)], "width": 2},
                marker={"size": 7, "color": palette[idx % len(palette)]},
                fill="toself" if scenario != "original" else None,
                opacity=0.35 if scenario != "original" else 0.9,
                customdata=np.column_stack([
                    np.array(raw_values + [raw_values[0]]),
                    np.array(scaled_values + [scaled_values[0]]),
                    np.array([scenario] * len(theta), dtype=object),
                    np.array(theta, dtype=object),
                ]),
                hovertemplate=(
                    "scenario=%{customdata[2]}<br>"
                    "feature=%{customdata[3]}<br>"
                    "raw=%{customdata[0]:.4f}<br>"
                    "standardized=%{customdata[1]:.4f}<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        title=title,
        width=width,
        height=height,
        template="plotly_white",
        hovermode="closest",
        polar={
            "radialaxis": {"visible": True, "range": [-radial_limit, radial_limit]},
            "angularaxis": {"direction": "clockwise"},
        },
        legend_title_text="Scenario",
    )
    fig.show()


## ケース比較

`risk_proba` の高・中・閾値近傍の 3 ケースで同じ分析を行い、得られたシナリオを比較する。

In [79]:
positive_candidate_df = scored_test_df.loc[
    (scored_test_df["predicted_risk"] == 1) & (scored_test_df[target_col] == 1)
].copy()

positive_candidate_df["distance_to_threshold"] = positive_candidate_df["risk_proba"] - best_threshold


def pick_case_index(candidate_df: pd.DataFrame, target_proba: float | None = None, mode: str = "nearest") -> int:
    if mode == "highest":
        return int(candidate_df["risk_proba"].idxmax())
    if target_proba is None:
        raise ValueError("target_proba is required unless mode='highest'")
    return int((candidate_df["risk_proba"] - target_proba).abs().idxmin())


case_definitions = [
    {"case_id": "high", "label": "High", "mode": "nearest", "target_proba": 0.85},
    {"case_id": "mid", "label": "Mid", "mode": "nearest", "target_proba": 0.70},
    {"case_id": "low", "label": "Low", "mode": "nearest", "target_proba": 0.55},
]

case_rows = []
for case in case_definitions:
    case_index = pick_case_index(
        positive_candidate_df,
        target_proba=case["target_proba"],
        mode=case["mode"],
    )
    case_rows.append(
        {
            "case_id": case["case_id"],
            "case_label": case["label"],
            "index": case_index,
            "risk_proba": float(positive_candidate_df.loc[case_index, "risk_proba"]),
            "distance_to_threshold": float(positive_candidate_df.loc[case_index, "distance_to_threshold"]),
        }
    )

case_summary_df = pd.DataFrame(case_rows)
display(case_summary_df)

selected_case_preview_df = (
    case_summary_df.set_index("case_label")[["index", "risk_proba", "distance_to_threshold"]]
    .join(model_test_df.loc[case_summary_df["index"], actionable_features].set_index(case_summary_df["case_label"]))
)
selected_case_preview_df


,case_id,case_label,index,risk_proba,distance_to_threshold
0,high,High,1608,0.849976,0.429976
1,mid,Mid,38466,0.699949,0.279949
2,low,Low,28390,0.549818,0.129818


,index,risk_proba,distance_to_threshold,dti_log1p,employment_length_ordinal,requested_loan_amount
case_label,,,,,,
High,1608,0.849976,0.429976,2.744704,0.0,15000.0
Mid,38466,0.699949,0.279949,3.434954,0.0,10000.0
Low,28390,0.549818,0.129818,2.864484,1.0,15000.0


In [80]:
desired_risk_class = 0

case_generation_config = {
    "name": "baseline_3cf_diverse",
    "features": actionable_features,
    "total_CFs": 3,
    "diversity_weight": 30.0,
    "proximity_weight": 0.2,
    "stopping_threshold": 0.50,
}


def generate_case_counterfactuals(case_index: int):
    case_query_scaled = model_test_scaled_df.loc[[case_index], feature_cols].copy()
    case_query_unscaled = model_test_df.loc[[case_index], feature_cols].copy()

    case_features = case_generation_config["features"]
    case_unscaled_range = build_permitted_range(model_train_df, case_query_unscaled, case_features)
    case_permitted_range = {
        col: [scale_feature_value(col, bounds[0]), scale_feature_value(col, bounds[1])]
        for col, bounds in case_unscaled_range.items()
    }

    case_counterfactuals = dice.generate_counterfactuals(
        query_instances=case_query_scaled,
        total_CFs=case_generation_config["total_CFs"],
        desired_class=desired_risk_class,
        features_to_vary=case_features,
        permitted_range=case_permitted_range,
        stopping_threshold=case_generation_config["stopping_threshold"],
        diversity_weight=case_generation_config["diversity_weight"],
        proximity_weight=case_generation_config["proximity_weight"],
    )

    case_cf_scaled = case_counterfactuals.cf_examples_list[0].final_cfs_df.copy()
    case_cf_unscaled = inverse_scale_features(case_cf_scaled)
    case_change_summary = summarize_changes(case_query_unscaled, case_cf_unscaled, case_features)
    return (
        case_query_scaled,
        case_query_unscaled,
        case_cf_scaled,
        case_cf_unscaled,
        case_change_summary,
        case_generation_config["name"],
    )

case_result_frames = []
case_risk_frames = []
case_status_rows = []
case_value_raw_frames = []
case_value_scaled_frames = []

for row in case_rows:
    try:
        (
            case_query_scaled,
            case_query_unscaled,
            case_cf_scaled,
            case_cf_unscaled,
            case_change_summary,
            strategy_name,
        ) = generate_case_counterfactuals(row["index"])
    except Exception as exc:
        case_status_rows.append(
            {
                "case_id": row["case_id"],
                "case_label": row["case_label"],
                "index": row["index"],
                "risk_proba": row["risk_proba"],
                "status": "failed",
                "strategy": "",
                "detail": str(exc),
            }
        )
        continue

    case_status_rows.append(
        {
            "case_id": row["case_id"],
            "case_label": row["case_label"],
            "index": row["index"],
            "risk_proba": row["risk_proba"],
            "status": "ok",
            "strategy": strategy_name,
            "detail": "",
        }
    )

    case_result = case_change_summary.copy()
    case_result.insert(0, "case_id", row["case_id"])
    case_result.insert(1, "case_label", row["case_label"])
    case_result.insert(2, "strategy", strategy_name)
    case_result_frames.append(case_result)

    case_value_raw, case_value_scaled = build_scenario_value_matrices(
        case_query_unscaled,
        case_cf_unscaled,
        case_query_scaled,
        case_cf_scaled,
        case_generation_config["features"],
    )
    case_value_raw.index = pd.MultiIndex.from_product(
        [[row["case_label"]], case_value_raw.index],
        names=["case_label", "scenario"],
    )
    case_value_scaled.index = pd.MultiIndex.from_product(
        [[row["case_label"]], case_value_scaled.index],
        names=["case_label", "scenario"],
    )
    case_value_raw_frames.append(case_value_raw)
    case_value_scaled_frames.append(case_value_scaled)

    case_risk_frames.append(
        pd.DataFrame(
            {
                "case_id": row["case_id"],
                "case_label": row["case_label"],
                "strategy": strategy_name,
                "scenario": ["original"] + [f"CF_{i}" for i in range(1, len(case_cf_scaled) + 1)],
                "risk_proba": [float(dice_ready_model.predict_proba(case_query_scaled)[:, 1][0])]
                + dice_ready_model.predict_proba(case_cf_scaled)[:, 1].tolist(),
            }
        )
    )

case_status_df = pd.DataFrame(case_status_rows)
display(case_status_df)

case_comparison_df = (
    pd.concat(case_result_frames, ignore_index=True)
    if case_result_frames else pd.DataFrame()
)
case_risk_df = (
    pd.concat(case_risk_frames, ignore_index=True)
    if case_risk_frames else pd.DataFrame()
)

if not case_comparison_df.empty:
    display(case_comparison_df.sort_values(["case_id", "cf_id", "abs_delta"], ascending=[True, True, False]).reset_index(drop=True))
    display(case_risk_df)


100%|██████████| 1/1 [00:03<00:00,  3.03s/it]


,case_id,case_label,index,risk_proba,status,strategy,detail
0,high,High,1608,0.849976,failed,,No counterfactuals found for any of the query ...
1,mid,Mid,38466,0.699949,ok,baseline_3cf_diverse,
2,low,Low,28390,0.549818,ok,baseline_3cf_diverse,


,case_id,case_label,strategy,cf_id,feature,before,after,delta,abs_delta
0,low,Low,baseline_3cf_diverse,CF_1,requested_loan_amount,15000.000000,2362.3,-12637.700000,12637.700000
1,low,Low,baseline_3cf_diverse,CF_1,dti_log1p,2.864484,2.4,-0.464484,0.464484
2,mid,Mid,baseline_3cf_diverse,CF_1,dti_log1p,3.434954,3.3,-0.134954,0.134954
3,mid,Mid,baseline_3cf_diverse,CF_2,dti_log1p,3.434954,3.2,-0.234954,0.234954
4,mid,Mid,baseline_3cf_diverse,CF_3,dti_log1p,3.434954,3.2,-0.234954,0.234954


,case_id,case_label,strategy,scenario,risk_proba
0,mid,Mid,baseline_3cf_diverse,original,0.699949
1,mid,Mid,baseline_3cf_diverse,CF_1,0.539523
2,mid,Mid,baseline_3cf_diverse,CF_2,0.316755
3,mid,Mid,baseline_3cf_diverse,CF_3,0.316755
4,low,Low,baseline_3cf_diverse,original,0.549818
5,low,Low,baseline_3cf_diverse,CF_1,0.396174


In [81]:
if not case_comparison_df.empty:
    case_delta_matrix = (
        case_comparison_df.pivot_table(
            index=["case_label", "cf_id"],
            columns="feature",
            values="delta",
            aggfunc="first",
        )
        .reindex(columns=actionable_features)
        .fillna(0)
    )

    case_value_raw_df = pd.concat(case_value_raw_frames).reindex(columns=actionable_features)
    case_value_scaled_df = pd.concat(case_value_scaled_frames).reindex(columns=actionable_features)

    # display(case_value_raw_df)
    display(
        style_raw_value_matrix(
            case_value_raw_df,
            case_value_scaled_df,
            caption="表示値と背景色は同じ生データ空間の値",
        )
    )

    case_labels = case_value_raw_df.index.get_level_values("case_label").unique()
    for case_label in case_labels:
        plot_scenario_value_dotplot(
            case_value_raw_df.loc[case_label],
            case_value_scaled_df.loc[case_label],
            title=f"{case_label} Case Scenarios (x: standardized delta from original)",
        )
        # plot_scenario_value_barplot(
        #     case_value_raw_df.loc[case_label],
        #     case_value_scaled_df.loc[case_label],
        #     title=f"{case_label} Case Scenarios (grouped raw-value bars)",
        # )

    # display(case_delta_matrix)

    case_max_abs_delta = case_delta_matrix.abs().to_numpy().max()
    if case_max_abs_delta == 0:
        case_max_abs_delta = 1.0

    display(
        case_delta_matrix.style
        .format("{:.4f}")
        .background_gradient(cmap="RdBu_r", axis=None, vmin=-case_max_abs_delta, vmax=case_max_abs_delta)
    )


In [82]:
# if not case_comparison_df.empty:
#     case_labels = case_value_raw_df.index.get_level_values("case_label").unique()
#     for case_label in case_labels:
#         plot_scenario_radar(
#             case_value_raw_df.loc[case_label],
#             case_value_scaled_df.loc[case_label],
#             title=f"{case_label} Case Radar Plot (standardized values)",
#         )


## メモ

単一ケース内の多様性と、ケース間で共通して動く特徴を確認する。